# 01 · Import and profile listings

## Goal
Confirm the source structure and inspect missing values, duplicate IDs, and numerical fields before changing any rows. See the [proposal](../docs/proposal.md) and [data setup](../DATA.md). Outputs report aggregates only.

## Snapshot findings
The pinned v10 file has **426,880 listings and 26 columns**. Listing IDs are populated and unique. There are **32,895 zero prices**, which are handled in notebook 02.

## Setup

In [1]:
from pathlib import Path
import sys

root = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src" / "listings.py").is_file()), None)
if root is None:
    raise RuntimeError("Start Jupyter from the repository or its notebooks directory.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from IPython.display import display

from src.listings import RAW, check_input, profile_source, read_source, finite_numeric

source = root / RAW
columns = check_input(source)
print(f"{RAW.as_posix()}: {source.stat().st_size / 1024**2:,.1f} MiB")


data/raw/vehicles.csv: 1,380.9 MiB


## Source schema and missingness
Count all fields in chunks. Missingness follows the CSV parser's missing-value rules before normalization.

In [2]:
rows, missingness = profile_source(source)
print(f"{rows:,} records × {len(columns)} fields")
display(missingness.round({"missing_pct": 2}).rename_axis("source field"))


426,880 records × 26 fields


,missing,missing_pct
source field,,
county,426880,100.00
size,306361,71.77
cylinders,177678,41.62
condition,174104,40.79
VIN,161042,37.73
drive,130567,30.59
paint_color,130203,30.50
type,92858,21.75
manufacturer,17646,4.13


## Listing IDs and numeric fields
Numeric summaries include zero and unusual values at this stage.

In [3]:
raw = read_source(source)
ids = raw["id"].str.strip().replace("", None)
print(f"Missing IDs: {ids.isna().sum():,}")
print(f"Repeated populated IDs beyond the first row: {(ids.notna() & ids.duplicated()).sum():,}")
numbers = raw[["price", "year", "odometer"]].apply(finite_numeric)
display(numbers.describe().T.round(2).rename(index={
    "price": "price (USD)", "year": "model year", "odometer": "odometer (mi)"
}))
print(f"Non-positive numeric prices: {numbers['price'].le(0).sum():,}")


Missing IDs: 0
Repeated populated IDs beyond the first row: 0


,count,mean,std,min,25%,50%,75%,max
price (USD),426880.0,75199.03,12182282.17,0.0,5900.0,13950.0,26485.75,3.736929e+09
model year,425675.0,2011.24,9.45,1900.0,2008.0,2013.0,2017.00,2.022000e+03
odometer (mi),422480.0,98043.33,213881.50,0.0,37704.0,85548.0,133542.50,1.000000e+07


Non-positive numeric prices: 32,895


## Next step
Run [02 · Clean and derive](02_clean_and_derive.ipynb) for the cleaning audit and local analysis table.